In [1]:
import pandas as pd

import nb_utils

In [2]:
ds = nb_utils.load_dataset()

In [3]:
df = ds.metadata.as_dataframe()
df.columns

Index(['Artic_primer_version', 'Collection_date', 'Country', 'Date_tree',
       'Date_tree_order', 'Experiment', 'First_created', 'Genbank_N',
       'Genbank_accession', 'Genbank_other_runs', 'Genbank_pangolin',
       'Genbank_scorpio', 'Genbank_tree_name', 'In_Viridian_tree',
       'In_intersection', 'In_may_2024_preprint', 'Platform', 'Region',
       'Run_count', 'Sample', 'Study', 'Viridian_N',
       'Viridian_amplicon_scheme', 'Viridian_cons_het', 'Viridian_cons_len',
       'Viridian_pangolin', 'Viridian_pangolin_1.29', 'Viridian_result',
       'Viridian_scorpio', 'Viridian_scorpio_1.29'],
      dtype='str')

In [4]:
# How many samples have an assigned collection date of 2020 New Year's Eve?
sub_df = df[df["Date_tree"] == "2020-12-31"]
len(sub_df)

40070

In [ ]:
# Downloaded from https://www.ebi.ac.uk/ena/browser/view/PRJEB37886
# Select: sample_accession, study_accession, run_accession, sample_alias
# Download report TSV to get filereport_read_run_PRJEB37886.tsv.
# 
# md5sum-lite
# 289eb899937e509944870f19a6037cf6  filereport_read_run_PRJEB37886.tsv
#
# Date accessed: October 21, 2026
ena_df = pd.read_csv("filereport_read_run_PRJEB37886.tsv", sep="\t")
ena_df["sample_name"] = ena_df["sample_alias"].str.removeprefix("COG-UK/")
ena_df

,run_accession,study_accession,sample_accession,sample_alias,sample_name
0,ERR10000002,PRJEB37886,SAMEA110427045,COG-UK/LSPA-3EBE8D5,LSPA-3EBE8D5
1,ERR10000004,PRJEB37886,SAMEA110427047,COG-UK/LSPA-3EBBC63,LSPA-3EBBC63
2,ERR10000010,PRJEB37886,SAMEA110427053,COG-UK/LSPA-3EBED21,LSPA-3EBED21
3,ERR10000015,PRJEB37886,SAMEA110427058,COG-UK/LSPA-3EBC01B,LSPA-3EBC01B
4,ERR10000020,PRJEB37886,SAMEA110427063,COG-UK/LSPA-3EBED5E,LSPA-3EBED5E
...,...,...,...,...,...
2700696,ERR9978024,PRJEB37886,SAMEA110400310,COG-UK/PLYM-3EE1851,PLYM-3EE1851
2700697,ERR9978043,PRJEB37886,SAMEA110400329,COG-UK/PLYM-3EE20FE,PLYM-3EE20FE
2700698,ERR9978045,PRJEB37886,SAMEA110400331,COG-UK/PLYM-3EE1E9B,PLYM-3EE1E9B
2700699,ERR9978048,PRJEB37886,SAMEA110400334,COG-UK/PLYM-3EE19F4,PLYM-3EE19F4


In [6]:
# Downloaded from https://datadryad.org/dataset/doi:10.5061/dryad.hx3ffbgm2
# Date accessed: October 21, 2026
coguk_df = pd.read_csv("SampleInfo.csv")
coguk_df

,sample_name,collection_date,pangolin_call,major_lineage,downsampled
0,MILK-A90FD1,2020-10-15,B.1.177.54,B.1.177,NaN
1,MILK-A913D4,2020-10-17,B.1.177,B.1.177,NaN
2,MILK-A914D1,2020-10-17,B.1.1.279,Other,NaN
3,MILK-A914E0,2020-10-16,AD.2,Other,NaN
4,MILK-A91A48,2020-10-17,B.1.177,B.1.177,NaN
...,...,...,...,...,...
126367,OXON-FA9C60,2021-07-13,AY.4,B.1.617.2,NaN
126368,OXON-FA9CCA,2021-07-12,AY.4,B.1.617.2,NaN
126369,OXON-FA9CD9,2021-07-13,AY.4,B.1.617.2,NaN
126370,OXON-FA9CE8,2021-07-13,AY.4,B.1.617.2,NaN


In [7]:
# Cross-reference Viridian metadata and ENA metadata
#sub_df, sample_id (actually, run accession), Collection_date
#ena_df, run_accession, sample_name

df_merged = pd.merge(
    sub_df,
    ena_df,
    left_on="sample_id",
    right_on="run_accession",
    how="left"
)[["run_accession", "Collection_date", "sample_name"]]
df_merged = df_merged[df_merged.run_accession.notna()].reset_index(drop=True)
df_merged

,run_accession,Collection_date,sample_name
0,ERR4365427,2020,LOND-124C7FF
1,ERR4365550,2020,LOND-124C874
2,ERR4365558,2020,LOND-124C7B3
3,ERR4365949,2020,LOND-124C81A
4,ERR4365954,2020,LOND-124C777
...,...,...,...
39168,ERR8188001,2020,PHEC-YYB4IQ3
39169,ERR8188022,2020,PHEC-4M070ZF7
39170,ERR8188023,2020,PHEC-4L0BDZ65
39171,ERR8188026,2020,PHEP-YYN9FTZ


In [8]:
# Cross-reference ENA metadata and COG-UK metadata
#ena_df, run_accession, sample_name
#coguk_df, sample_name, collection_date

df_merged = pd.merge(
    df_merged,
    coguk_df,
    left_on="sample_name",
    right_on="sample_name",
    how="left"
)[["run_accession", "Collection_date", "sample_name", "collection_date"]]
df_merged = df_merged[df_merged.collection_date.notna()].reset_index(drop=True)
df_merged

,run_accession,Collection_date,sample_name,collection_date
0,ERR6486792,2020,PHEC-341325,2021-07-25
1,ERR6494151,2020,PHEC-341422,2021-07-28
2,ERR6754676,2020,PHEC-YYBDGM7,2021-08-13
3,ERR6756803,2020,PHEC-YYBRSGI,2021-08-30
4,ERR6825090,2020,PHEC-YYBJ4YU,2021-09-07
...,...,...,...,...
2034,ERR7375585,2020,NORT-YYTWQP,2021-11-03
2035,ERR7376210,2020,NORT-YYTWUN,2021-11-04
2036,ERR7376228,2020,NORT-YYTEHH,2021-11-04
2037,ERR7378417,2020,NORT-YYT15W,2021-11-04


In [9]:
date_ena = pd.to_datetime(df_merged['Collection_date'], errors='coerce')
date_coguk = pd.to_datetime(df_merged['collection_date'], errors='coerce')

mask = date_ena.dt.year != date_coguk.dt.year

print(f"Different year: {mask.sum()}")
print(f"Percentage: {mask.mean() * 100:.2f}%")

Different year: 2039
Percentage: 100.00%


In [10]:
cofreq = pd.crosstab(
    date_ena.dt.year,
    date_coguk.dt.year,
    rownames=['Collection_date'],
    colnames=['collection_date']
)

display(cofreq)

collection_date,2021
Collection_date,
2020,2039
